In [ ]:
import numpy as np 
import pandas as pd

movies=pd.read_csv('final.csv')
movies[movies.duplicated(subset='id', keep=False)]
movies.drop_duplicates(subset='id', keep='first', inplace=True)

In [ ]:

from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

def stem(text):
    l=[]
    for i in text.split():
        l.append(ps.stem(i))
    string=" ".join(l)
    return string

In [ ]:
import re
movies['genres'] = movies['genres'].fillna('').astype(str).apply(
    lambda x: re.sub(r'[^\w\s]', '', x).lower()
)
genres=movies[['id','genres']]
genres.loc[:, 'genres'] = genres['genres'].apply(stem)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv_genres=CountVectorizer(stop_words='english')
genres_ind=cv_genres.fit_transform(genres['genres'])

genres_count=genres_ind.toarray().sum(axis=0)
genres_names=cv_genres.get_feature_names_out()
genres_freq=list(zip(genres_names,genres_count))

import os
os.makedirs('genres',exist_ok=True)

top_genres=sorted(genres_freq, key= lambda x:x[1],reverse=True)[:100]
df_top = pd.DataFrame(top_genres, columns=['genres', 'count'])
df_top['rank'] = df_top.index
df_top[['rank', 'genres']].to_csv('genres/top_100_genres.csv', index=False)

In [31]:
top_genres=df_top['genres'].tolist()

cv_top=CountVectorizer(vocabulary=top_genres)
top_ind=cv_top.transform(genres['genres'])

top_dense=top_ind.toarray()
np.savez_compressed('genres/movie_genres_mapping.npz',top_dense)


#To read the array
data = np.load('genres/movie_genres_mapping.npz')
arr = data['arr_0'] 

print("Shape of array:", arr.shape)


Shape of array: (3000, 21)


In [32]:
loaded = np.load('final/final_movies.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])
loaded = np.load('final/final_similarity.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])

[2, 4, 10, 5, 34, 268, 6, 385, 58, 150, 86, 589, 7, 72, 190, 250, 12, 97, 288, 0, 196, 112, 1917, 840, 17, 39, 169, 82, 191, 1451, 377, 54, 119, 223, 27, 36, 194, 454, 152, 286, 244, 364, 30, 44, 256, 103, 50, 149, 239, 305, 433, 751, 2217, 28, 42, 224, 13, 88, 242, 68, 301, 350, 426, 572, 734, 1055, 11, 284, 219, 457, 2500, 140, 235, 271, 298, 19, 142, 187, 330, 336, 343, 381, 441, 151, 320, 345, 8, 84, 118, 20, 75, 76, 101, 45, 59, 92, 128, 307, 64, 91, 107, 183, 380, 463, 528, 961, 153, 37, 43, 57, 65, 67, 87, 94, 98, 99, 102, 111, 124, 148, 161, 174, 185, 192, 207, 213, 228, 243, 252, 253, 262, 263, 306, 317, 326, 332, 346, 367, 378, 383, 386, 404, 425, 438, 442, 459, 953, 514, 15, 70, 166]
[np.float64(12.5), np.float64(11.5), np.float64(8.5), np.float64(8.5), np.float64(7.5), np.float64(7.5), np.float64(7.0), np.float64(7.0), np.float64(7.0), np.float64(6.5), np.float64(6.5), np.float64(6.5), np.float64(6.5), np.float64(6.0), np.float64(6.0), np.float64(6.0), np.float64(6.0), np.f

In [33]:
top_k = 300
top_similar_movies = []

final_similarity = np.load('final/final_similarity.npz', allow_pickle=True)['arr_0']
final_movies = np.load('final/final_movies.npz', allow_pickle=True)['arr_0']

updated_similarities = []
updated_movies = []

os.makedirs('genres', exist_ok=True)


for i in range(top_dense.shape[0]):
    cast_i=top_dense[i]
    similarities=[]

    existing_movies=list(final_movies[i])
    existing_sims=list(final_similarity[i])
    sim_dict={movie_id:sim for movie_id,sim in zip(existing_movies,existing_sims)}
    

    for j in range (top_dense.shape[0]):
        if i==j:
            continue

        cast_j = top_dense[j]  # Replace this with `genres_dense[j]` if available
        sim = np.sum(np.minimum(cast_i, cast_j))

        if i==1 and j==2:
            print (sim)
        if i==1 and j==385:
            print (sim)
            
        if sim <= 0:
            continue

        if j in sim_dict:
            sim_dict[j]+=sim/2
        elif len(sim_dict)<top_k:
            sim_dict[j]=sim/2
            
    sorted_sims = sorted(sim_dict.items(), key=lambda x: x[1], reverse=True)[:top_k]
    updated_movies.append([j for j, _ in sorted_sims])
    updated_similarities.append([sim for _, sim in sorted_sims])
            
    
    # Pad with -1 if fewer than top_k
    padded = [j for j, _ in sorted_sims] + [-1] * (top_k - len(sorted_sims))
    top_similar_movies.append(padded)
np.savez_compressed('genres/top_300_similar_movies.npz', top_similar_movies)
np.savez_compressed('final/final_similarity.npz', np.array(updated_similarities, dtype=object))
np.savez_compressed('final/final_movies.npz', np.array(updated_movies, dtype=object))

4
4


In [34]:
loaded = np.load('genres/top_300_similar_movies.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[0]
print(first_row)


[  20    5 1523  239  110  560    1    7   34  385    6   10  235  268
  191  325  196   50  648 1517   28   39   12  169  350   27   13   68
  330  176  257   36  194   82  152  384  127  207  262  537  576  302
  154    4   30   42   97  150  224  284   11   17  101  256  454   77
  317  535  579  107  137   88  298   86  332  491  578  587  215   44
   76  118  132  346   94  124  161  243  305  402  404  412  452  523
  559  586  572    8   84   91   72   75  140  242  287   45  128  144
  149  213  221  343  375  516  575  146    3  103  445   64  166  307
  148  160  175  252  263  386  433  459  547  219  228  250  271  366
  377  557   52  117  142  187  226  286  367  380  383  438  442  457
  467  533  545   43  126  190    2  369   31   57  111  122  155  185
  197  275  381  415  423  473  493  526  558  410  209   24   67   70
   93  108  129  130  133  174  193  202  236  269  283  285  334  364
  376  378  389  393  408  411  422  426  435  449  463  464  479  511
  538 

In [40]:
loaded = np.load('final/final_movies.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
for idx in first_row[:50]:
    print(movies.loc[idx, 'title'])
loaded = np.load('final/final_similarity.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])

Avengers: Endgame
Captain America: Civil War
Thor: Ragnarok
Black Panther
Spider-Man: Into the Spider-Verse
The Flash
Sonic the Hedgehog 2
Spider-Man: Homecoming
Doctor Strange
Black Widow
Spider-Man: No Way Home
Black Adam
Fantastic Beasts: The Crimes of Grindelwald
Men in Black: International
Ant-Man and the Wasp
Dark Phoenix
Thor: Love and Thunder
Extraction 2
Doctor Strange in the Multiverse of Madness
Deadpool
Ghost in the Shell
Sonic the Hedgehog
Lightyear
Black Panther: Wakanda Forever
Jojo Rabbit
Extraction
Team Thor: Part 2
Batman v Superman: Dawn of Justice
Venom
Spider-Man: Far From Home
X-Men: Apocalypse
Rampage
Sing 2
No Time to Die
Team Thor
Aquaman
Puss in Boots: The Last Wish
Captain Marvel
Justice League
Guardians of the Galaxy Vol. 3
Teenage Mutant Ninja Turtles: Out of the Shadows
Cherry
Shazam! Fury of the Gods
The Huntsman: Winter's War
Spider-Man: Across the Spider-Verse
Glass
Logan
Dune
Ant-Man and the Wasp: Quantumania
Bloodshot
[np.float64(14.5), np.float64(13.